In [ ]:
import os
import sys
sys.path.insert(0, os.path.realpath('__file__'))

import numpy as np
import matplotlib.pyplot as plt
from sdypy import EMA

The experiment is shown in the figure below; the beam was excited at 6 locations with an impact hammer, while the response was measured at 7 locations using piezo accelerometers (camera data is not used in this showcase; for image EMA, see https://github.com/ladisk/ImageBasedModalAnalysisTutorial):
<img width=500 src="./data/experiment_1.jpg">

Loading experimental data:

In [ ]:
freq, H1_main = np.load("./data/acc_data.npy", allow_pickle=True)

Frequency vector was truncated for zero frequency:

In [ ]:
freq[:10]

And from the FRF (`H1_main` is of dimensions: #inputs, #outputs, frequency) only the response accelerometer position at index 1 will be used later:

In [ ]:
selected_response = 1
FRF = H1_main[:,selected_response,:]

In [ ]:
plt.semilogy(freq, np.abs(FRF.T));
plt.xlim(0,1000);

Initiate the pyEMA object:

In [ ]:
acc = EMA.Model(frf=FRF, 
                 freq=freq,
                 lower=10, 
                 upper=5000,
                 pol_order_high=60)

Get the poles:

In [ ]:
acc.get_poles()

Select stable poles from the stabilisation chart:

In [ ]:
acc.select_poles()

As an alternative to selecting from the stabilisation chart, the frequencies can be defined as a list, e.g. (comment out to use):

In [ ]:
#n_freq =  [176,476,932,1534,2258,3161,4180]
#acc.select_closest_poles(n_freq)

Reconstruction of FRF

In [ ]:
frf_rec, modal_const = acc.get_constants(whose_poles='own', FRF_ind='all', upper_r=False)

The selected natural frequencies and corresponding damping factors:

In [ ]:
acc.print_modal_data()

Modal constants are of shape (`n_locations` $\times$ `n_frequencies`):

In [ ]:
acc.A.shape

In [ ]:
acc.A[:, 0]

Show normal modes (first three):

In [ ]:
plt.plot(acc.normal_mode()[:, :3]);

Upper and lower residuals can also be called:

In [ ]:
acc.UR, acc.LR

Check out the Auto Modal Assurance Criterion.

In [ ]:
auto_mac = acc.auto_mac()

In [ ]:
plt.matshow(auto_mac);

### Old and new names

sdypy-EMA 0.31 renamed `frf_type` to `frf_form` and `autoMAC` to `auto_mac`, following the SDyPy nomenclature in [SEP 2](https://github.com/sdypy/sdypy/blob/main/docs/seps/sep-0002.rst). The old names still work and emit a `DeprecationWarning`; they will be removed in the next major version. This cell only shows the mapping and is to be deleted together with the aliases.

In [ ]:
import warnings

with warnings.catch_warnings():
    warnings.simplefilter('always', DeprecationWarning)

    # Model: `frf_form` (new) replaces `frf_type` (old)
    acc_new = EMA.Model(frf=FRF, freq=freq, lower=10, upper=5000, frf_form='accelerance')
    acc_old = EMA.Model(frf=FRF, freq=freq, lower=10, upper=5000, frf_type='accelerance')  # DeprecationWarning
    print(acc_new.frf_form, acc_old.frf_form)

    # `auto_mac` (new) replaces `autoMAC` (old)
    print(np.array_equal(acc.auto_mac(), acc.autoMAC()))  # DeprecationWarning

Showing the reconstructed FRF

In [ ]:
freq_a = acc.freq
select_loc = 0

plt.figure(figsize = ((10,6)))
plt.subplot(211)

plt.semilogy(freq, np.abs(FRF[select_loc]), label='Experiment')
plt.semilogy(freq_a, np.abs(frf_rec[select_loc]),'--', label='LSCF')
plt.xlim(0,freq[-1])
plt.ylabel(r"abs($\alpha$)")

plt.legend(loc = 'best')

plt.subplot(212)
plt.plot(freq, np.angle(FRF[select_loc],deg = 1), label='Experiment')
plt.plot(freq_a, np.angle(frf_rec[select_loc],deg = 1),'--',label='LSCF')
plt.xlim(0,freq[-1])

plt.ylabel(r"angle($\alpha$)")
plt.legend(loc = 'best');